## Train policy based on min Q

In [7]:
using Pkg
using Revise

Pkg.activate("/home/jiaxingl/project/verify_julia_env")
# Pkg.update("ModelVerification")
Pkg.status()



  Activating project at `~/project/verify_julia_env`


Status `~/project/verify_julia_env/Project.toml`
  [d8c2afa5] Cersyve v1.0.0-DEV `~/project/Cersyve.jl`
⌅ [587475ba] Flux v0.13.17
  [f67ccb44] HDF5 v0.17.2
  [7073ff75] IJulia v1.26.0
⌅ [033835bb] JLD2 v0.4.53
⌅ [b4f0291d] LazySets v1.59.1
  [6d061d49] ModelVerification v0.1.0 `~/project/ModelVerification.jl`
  [85610aed] NaiveNASflux v2.0.8 `~/project/ModelVerification.jl/onnx_parser/NaiveNASflux`
⌃ [bd45eb3e] NaiveNASlib v2.0.11
⌅ [d0dd6a25] ONNX v0.2.0
  [2e935253] ONNXNaiveNASflux v0.2.7 `~/project/ModelVerification.jl/onnx_parser/ONNXNaiveNASflux`
⌅ [3bd65402] Optimisers v0.2.20
  [49802e3a] ProgressBars v1.5.1
  [438e738f] PyCall v1.96.4
⌅ [295af30f] Revise v3.6.4
⌅ [2913bbd2] StatsBase v0.33.21
⌃ [899adc3e] TensorBoardLogger v0.1.19
⌅ [e88e6eb3] Zygote v0.6.73
  [37e2e46d] LinearAlgebra
  [56ddb016] Logging
  [de0858da] Printf
  [9a3f8284] Random
  [10745b16] Statistics v1.9.0
Info Packages marked with ⌃ and ⌅ have new versions available, but those with ⌅ are restricted by comp

In [8]:
using Revise
using Cersyve
using Flux
using JLD2
using Random




In [9]:
# task = Unicycle
# task = LaneKeep
# task = DoubleIntegrator
# task= Pendulum
# task = CartPole
# task = Quadrotor
# task = PointMass
# task = DoubleIntegrator2D
task = Unicycle4D
# task = RobotArm
# task = TwoLinkRobotArm


value_hidden_sizes = [32, 32]
dynamics_hidden_sizes = [32, 32]
constraint_hidden_sizes = [16]
policy_hidden_sizes = [32, 32]


# data_path = joinpath(@__DIR__, "../data/2link_robot_arm_data.jld2")
# model_dir = joinpath(@__DIR__, "../model/2link_robot_arm/")
# log_dir = joinpath(@__DIR__, "../log/2link_robot_arm/")
# Q_path = "/home/jiaxingl/project/Cersyve.jl/log/2link_robot_arm/pretrain_Q_20250218_122211_old_argminQ/Q_pretrain.jld2"
# Q_path = "/home/jiaxingl/project/Cersyve.jl/log/2link_robot_arm/pretrain_Q_pi20250315_225155/Q_pretrain.jld2"
# Q_path ="/home/jiaxingl/project/Cersyve.jl/log/2link_robot_arm/finetune_20250218_160454_x_mul_xu_tol1e-4_verified/Q_finetune_final.jld2"

# data_path = joinpath(@__DIR__, "../data/double_intergrator_data.jld2")
# model_dir = joinpath(@__DIR__, "../model/double_integrator/")
# log_dir = joinpath(@__DIR__, "../log/double_integrator/")
# Q_path = "/home/jiaxingl/project/Cersyve.jl/log/double_integrator/finetune_20250316_190553_hold/Q_finetune_final.jld2"


# data_path = joinpath(@__DIR__, "../data/2D_double_integrator_data.jld2")
# model_dir = joinpath(@__DIR__, "../model/2D_double_integrator/")
# log_dir = joinpath(@__DIR__, "../log/2D_double_integrator/")
# Q_path = "/home/jiaxingl/project/Cersyve.jl/log/2D_double_integrator/finetune_20250211_192942_hold/Q_finetune_final.jld2"

data_path = joinpath(@__DIR__, "../data/unicycle4D_data.jld2")
model_dir = joinpath(@__DIR__, "../model/unicycle4D/")
log_dir = joinpath(@__DIR__, "../log/unicycle4D/")
Q_path = "/home/jiaxingl/project/Cersyve.jl/log/unicycle4D/pretrain_Q_20250321_151806_single_x_mul_xu/Q_pretrain.jld2"

# if not exist create folder
if !isdir(log_dir)
    mkdir(log_dir)
end

if !isdir(model_dir)
    mkdir(model_dir)
end

seed = 1

Random.seed!(seed)

# if !isfile(data_path)
#     collect_data(
#         task.x_low,
#         task.x_high,
#         task.u_low,
#         task.u_high,
#         task.dynamics,
#         task.terminated;
#         save_path=data_path,
#     )
# end


# non linear dynamic
data = JLD2.load(data_path)["data"]
f_model = Cersyve.create_mlp(task.x_dim + task.u_dim, task.x_dim, dynamics_hidden_sizes)
Flux.loadmodel!(f_model, JLD2.load(joinpath(model_dir, "f.jld2"), "state"))
f_pi_model = Cersyve.create_closed_loop_dynamics_model(
    f_model, task.pi_model, data, task.x_low, task.x_high, task.u_dim)
dynamics_model = Cersyve.create_non_linear_dynamics_model(f_model, data, task.x_low, task.x_high, task.u_dim, task.x_dim)


h_model = Cersyve.create_mlp(task.x_dim, 1, constraint_hidden_sizes)
# train_constraint(task.x_low, task.x_high, h_model, task.constraint, log_dir=log_dir)
# Flux.loadmodel!(h_model, JLD2.load(joinpath(model_dir, "h_double_circle.jld2"), "state"))
Flux.loadmodel!(h_model, JLD2.load(joinpath(model_dir, "h.jld2"), "state"))




x_a_low =  [task.x_low; task.u_low]
x_a_high = [task.x_high; task.u_high]


# affine_Q = create_parallel_affine_Q(task.x_dim, task.u_dim)
# affine_Q = create_mul_Q(task.x_dim, task.u_dim)
# affine_Q = create_lim_X_affine_Q(task.x_dim, task.u_dim)
# affine_Q = create_baseline_affine_Q(task.x_dim, task.u_dim)
affine_Q = create_x_mul_xu_Q(task.x_dim, task.u_dim)
# affine_Q = create_x_add_xu_Q(task.x_dim, task.u_dim)


Flux.loadmodel!(affine_Q, JLD2.load(Q_path, "state"))

# pi_model = Cersyve.create_mlp(task.x_dim, task.u_dim, policy_hidden_sizes)
# pi_path = "/home/jiaxingl/project/Cersyve.jl/log/double_integrator/policy_20250316_181631/pi_pretrained.jld2"
pi_model = create_policy(task.x_dim, task.u_dim, policy_hidden_sizes, task)
# Flux.loadmodel!(pi_model, JLD2.load(pi_path, "state"))

train_policy(
    pi_model,
    dynamics_model,
    h_model,
    affine_Q,
    task.x_low,
    task.x_high,
    task.u_low,
    task.u_high,
    task;
    pi_ready = false,
    gamma = 0.9,
    lr = 3e-4,
    batch_size = 256,
    iter_num = 10000,
    weight_decay = 1e-4,
    log_dir = log_dir,
    q_step = 0,
)

# train_policy(
#     pi_model,
#     task.f_model,
#     h_model,
#     affine_Q,
#     task.x_low,
#     task.x_high,
#     task.u_low,
#     task.u_high,
#     task;
#     pi_ready = false,
#     gamma = 0.9,
#     lr = 3e-4,
#     batch_size = 256,
#     iter_num = 10000,
#     weight_decay = 1e-4,
#     log_dir = log_dir,
#     q_step = 0,
# )



length of layers:3
layers of model:7


0.0%┣                                            ┫ 0/10.0k [00:00<00:00, -0s/it]
0.0%┣                                        ┫ 1/10.0k [00:06<Inf:Inf, InfGs/it]
0.0%┣                                          ┫ 3/10.0k [00:06<08:47:46, 3s/it]
0.1%┣                                          ┫ 7/10.0k [00:06<02:58:36, 1s/it]
0.1%┣                                          ┫ 8/10.0k [00:07<02:35:59, 1it/s]
0.1%┣                                         ┫ 11/10.0k [00:07<01:50:23, 2it/s]
0.1%┣                                         ┫ 15/10.0k [00:07<01:19:52, 2it/s]
0.2%┣                                         ┫ 19/10.0k [00:07<01:03:03, 3it/s]
0.2%┣                                            ┫ 22/10.0k [00:07<54:58, 3it/s]
0.3%┣▏                                           ┫ 26/10.0k [00:07<46:41, 4it/s]
0.3%┣▏                                           ┫ 31/10.0k [00:07<39:12, 4it/s]
0.3%┣▏                                           ┫ 33/10.0k [00:07<37:02, 4it/s]
0.4%┣▏                      

In [10]:
ENV["GRB_LICENSE_FILE"] = "/home/jiaxingl/gurobi_lic/gurobi.lic"
ENV["GUROBI_HOME"] = "/home/jiaxingl/gurobi1101/linux64"

# Verify by printing the environment variables
println(ENV["GRB_LICENSE_FILE"])
println(ENV["GUROBI_HOME"])

In [ ]:
seed = 1

Random.seed!(seed)
# data = JLD2.load(data_path)["data"]
# f_model = Cersyve.create_mlp(task.x_dim + task.u_dim, task.x_dim, dynamics_hidden_sizes)
# Flux.loadmodel!(f_model, JLD2.load(joinpath(model_dir, "f.jld2"), "state"))
# f_pi_model = Cersyve.create_closed_loop_dynamics_model(
#     f_model, task.pi_model, data, task.x_low, task.x_high, task.u_dim)

# h_model = Cersyve.create_mlp(task.x_dim, 1, constraint_hidden_sizes)
# Flux.loadmodel!(h_model, JLD2.load(joinpath(model_dir, "h.jld2"), "state"))

x_a_low =  [task.x_low; task.u_low]
x_a_high = [task.x_high; task.u_high]

# fun_affine_Q = create_func_parallel_affine_Q(task.x_dim, task.u_dim)
# affine_Q = create_parallel_affine_Q(task.x_dim, task.u_dim)
# affine_Q = create_mul_affine_Q(task.x_dim, task.u_dim)
# affine_Q = create_mul_Q(task.x_dim, task.u_dim)
# affine_Q = create_baseline_affine_Q(task.x_dim, task.u_dim)
affine_Q = create_x_mul_xu_Q(task.x_dim, task.u_dim)
# affine_Q = create_x_add_xu_Q(task.x_dim, task.u_dim)
# Flux.loadmodel!(affine_Q, JLD2.load(joinpath(model_dir, "Q_pretrain.jld2"), "state"))
# Flux.loadmodel!(affine_Q, JLD2.load("/home/jiaxingl/project/Cersyve.jl/log/double_integrator/pretrain_Q_20250123_184057/Q_pretrain.jld2", "state"))

# mul_Q finetune
# Flux.loadmodel!(affine_Q, JLD2.load("/home/jiaxingl/project/Cersyve.jl/model/double_integrator/Q_mul_pretrain.jld2", "state"))
# Q_path = "/home/jiaxingl/project/Cersyve.jl/log/2link_robot_arm/finetune_20250218_160454_x_mul_xu_tol1e-4_verified/Q_finetune_final.jld2"
# Q_path = "/home/jiaxingl/project/Cersyve.jl/log/2link_robot_arm/pretrain_Q_pi20250315_225155/Q_pretrain.jld2"

Q_path = "/home/jiaxingl/project/Cersyve.jl/log/unicycle4D/pretrain_Q_20250321_151806_single_x_mul_xu/Q_pretrain.jld2"
Flux.loadmodel!(affine_Q, JLD2.load(Q_path, "state"))

# load policy
pi_path = "/home/jiaxingl/project/Cersyve.jl/log/unicycle4D/policy_20250321_162116/pi_pretrained.jld2"
pi_model = create_policy(task.x_dim, task.u_dim, policy_hidden_sizes, task)
saved_state = JLD2.load(pi_path, "state")


# finetune_Q_pi(
#     task, 
#     affine_Q,
#     task.f_model,
#     pi_model,
#     # task.pi_model,
#     h_model,
#     x_a_low,
#     x_a_high;
#     sample_size=100,
#     search_stop=1000,
#     search_size=3000,
#     log_dir=log_dir,
#     save_every=500,
#     early_stop = 0,
#     lr=1e-4,
#     tol = 0.0,
#     max_skip = 500,
#     pi_step = 50
# )

finetune_Q_pi(
    task, 
    affine_Q,
    dynamics_model,
    pi_model,
    # task.pi_model,
    h_model,
    x_a_low,
    x_a_high;
    sample_size=100,
    search_stop=1000,
    search_size=3000,
    log_dir=log_dir,
    save_every=500,
    early_stop = 0,
    lr=1e-4,
    tol = 0.0,
    max_skip = 500,
    pi_step = 50
)
    